# Knowledge Graph RAG — Demo
## Insolvencia Empresarial Colombiana

Este notebook demuestra las funcionalidades del sistema RAG agéntico:
1. Ingesta y vector store
2. Knowledge Graph (ontología + SPARQL)
3. Transformación de consultas (HyDE, Decomposition)
4. Agente ReAct + Reflecting
5. Métricas de evaluación
6. LLM como juez

In [2]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

## 1. Vector Store — Verificar ingesta

In [3]:
from src.ingestion.vector_store import load_vector_store, get_mmr_retriever

store = load_vector_store()
count = store._collection.count()
print(f'Chunks en ChromaDB: {count}')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7291.19it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks en ChromaDB: 4525


In [4]:
# Búsqueda MMR de ejemplo
retriever = get_mmr_retriever(store, k=5)
docs = retriever.invoke('requisitos para proceso de reorganización empresarial')
for i, doc in enumerate(docs, 1):
    print(f'[{i}] {doc.metadata["source"]} p.{doc.metadata["page"]}')
    print(f'    {doc.page_content[:150]}...\n')

[1] 90022.pdf p.15
    EL proceso de reorganización pretende, a través de un acuerdo, preservar 
empresas viables y normalizar sus relaciones comerciales y crediticias, medi...

[2] 90022.pdf p.52
    Los procesos concursales  y acuerdos de reorganización empresarial, 
Leyer 2005...

[3] Cartilla Insolvencia .pdf p.10
    INICIO Y TRÁMITE DEL PROCESO
La reorganización empresarial comienza con el auto (oficio de inicio para PRES) de iniciación del proceso, que no será 
s...

[4] 90022.pdf p.19
    CLASES 
 
2.1.1. REORGANIZACION EMPRESARIAL 
Destinada a superar las dificultades para lograr su viabilidad. Se formaliza en el documento que debe red...

[5] msms_insolvency_ebook_es.pdf p.9
    . 30
Recomendación 353. Conversión de una reorganización simplificada  
en una liquidación . ....



## 2. Knowledge Graph — Consultas SPARQL

In [5]:
from src.kg.ontology_manager import load_graph, query_graph

g = load_graph()
print(f'Triples en ontología: {len(g)}')

[INFO] Ontologia cargada: 356 triples.
Triples en ontología: 356


In [6]:
# SELECT + ORDER BY + LIMIT
results = query_graph(g, '''
    PREFIX ins: <http://www.unal.edu.co/ontologies/insolvencia#>
    SELECT ?norma ?numero ?anio WHERE {
        ?norma a ?tipo .
        VALUES ?tipo { ins:Ley ins:Decreto }
        ?norma ins:numeroNorma ?numero .
        OPTIONAL { ?norma ins:anoExpedicion ?anio }
    }
    ORDER BY ?anio
    LIMIT 10
''')
for r in results:
    print(f'  Norma {r["numero"]} — Año {r["anio"]}')

  Norma 1116 — Año 2006
  Norma 1564 — Año 2012
  Norma 806 — Año 2020
  Norma 772 — Año 2020
  Norma 2445 — Año 2025


In [7]:
# SELECT + FILTER
results = query_graph(g, '''
    PREFIX ins: <http://www.unal.edu.co/ontologies/insolvencia#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    SELECT ?proc ?fecha WHERE {
        ?proc a ins:Reorganizacion .
        ?proc ins:fechaAdmision ?fecha .
        FILTER (?fecha > "2023-01-01"^^xsd:date)
    }
    ORDER BY ?fecha
''')
for r in results:
    print(f'  {r["proc"].split("#")[-1]} — {r["fecha"]}')

  ReorgEjemplo1 — 2023-03-15
  ReorgEjemplo2 — 2024-01-10


In [8]:
# Herramientas KG del agente
from src.kg.sparql_tools import query_kg_norms, query_kg_procedures, query_kg_entity

print(query_kg_norms.invoke('1116'))
print()
print(query_kg_procedures.invoke('Empresa ABC S.A.S.'))
print()
print(query_kg_entity.invoke('Ley1116_2006'))

[INFO] Ontologia cargada: 356 triples.
Normas relacionadas con '1116':
  - Ley 1116 de 2006 (Num: 1116, Año: 2006)

Procedimientos para 'Empresa ABC S.A.S.':
  - Reorganizacion Ejemplo 1 - Empresa ABC (tipo: Reorganizacion, admision: 2023-03-15)

Propiedades de 'Ley1116_2006':
  - type: Ley
  - numeroNorma: 1116
  - anoExpedicion: 2006
  - label: Ley 1116 de 2006
  - comment: Regimen de Insolvencia Empresarial de Colombia.


## 3. Casos de Inferencia (5 casos)

In [9]:
# Caso 1: SubClass — buscar Deudor
r = query_graph(g, 'PREFIX ins: <http://www.unal.edu.co/ontologies/insolvencia#> SELECT ?x WHERE { ?x a ins:Deudor . }')
print(f'Caso 1 (subClassOf Deudor): {len(r)} resultados sin inferencia (0 esperado, 8 con GraphDB)')

# Caso 2: inverseOf — esAcreedorDe
r = query_graph(g, 'PREFIX ins: <http://www.unal.edu.co/ontologies/insolvencia#> SELECT ?a ?d WHERE { ?a ins:esAcreedorDe ?d . }')
print(f'Caso 2 (inverseOf esAcreedorDe): {len(r)} sin inferencia (0 esperado, 6 con GraphDB)')

# Caso 3: inverseOf — esIniciadoPor
r = query_graph(g, 'PREFIX ins: <http://www.unal.edu.co/ontologies/insolvencia#> SELECT ?p ?d WHERE { ?p ins:esIniciadoPor ?d . }')
print(f'Caso 3 (inverseOf esIniciadoPor): {len(r)} sin inferencia (0 esperado, 3 con GraphDB)')

# Caso 4: subPropertyOf — tieneAcreedorPrivilegiado
r = query_graph(g, 'PREFIX ins: <http://www.unal.edu.co/ontologies/insolvencia#> SELECT ?d ?a WHERE { ?d ins:tieneAcreedor ?a . }')
print(f'Caso 4 (subPropertyOf): {len(r)} explícitos (+2 inferidos con GraphDB)')

# Caso 5: equivalentClass unionOf
r = query_graph(g, 'PREFIX ins: <http://www.unal.edu.co/ontologies/insolvencia#> SELECT ?x WHERE { ?x a ins:Deudor . }')
print(f'Caso 5 (unionOf Deudor=PJ∪PN): {len(r)} sin inferencia (8 con GraphDB)')

Caso 1 (subClassOf Deudor): 0 resultados sin inferencia (0 esperado, 8 con GraphDB)
Caso 2 (inverseOf esAcreedorDe): 0 sin inferencia (0 esperado, 6 con GraphDB)
Caso 3 (inverseOf esIniciadoPor): 0 sin inferencia (0 esperado, 3 con GraphDB)
Caso 4 (subPropertyOf): 4 explícitos (+2 inferidos con GraphDB)
Caso 5 (unionOf Deudor=PJ∪PN): 0 sin inferencia (8 con GraphDB)


## 4. Transformación de Consultas

**Requiere OPENAI_API_KEY en .env**

In [10]:
from src.query.router import route_query
from src.query.hyde import apply_hyde
from src.query.decomposer import decompose_query
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Router
q1 = '¿Qué establece el artículo 9 de la Ley 1116?'
print(f'Router: "{q1}" → {route_query(llm, q1)}')

q2 = '¿Qué pasa cuando no se paga?'
print(f'Router: "{q2}" → {route_query(llm, q2)}')

q3 = '¿Cuáles son los requisitos para reorganización y las diferencias con la liquidación?'
print(f'Router: "{q3}" → {route_query(llm, q3)}')

Router: "¿Qué establece el artículo 9 de la Ley 1116?" → DIRECT
Router: "¿Qué pasa cuando no se paga?" → HYDE
Router: "¿Cuáles son los requisitos para reorganización y las diferencias con la liquidación?" → DECOMPOSE


In [11]:
# HyDE
hyde_doc = apply_hyde(llm, '¿Qué pasa cuando no se paga?')
print('Documento hipotético generado:')
print(hyde_doc)

Documento hipotético generado:
De acuerdo con lo establecido en el artículo 24 de la Ley 1116 de 2006, en caso de incumplimiento de las obligaciones de pago por parte del deudor en el marco del proceso de reorganización, el juez podrá declarar la terminación del proceso de reorganización y la apertura del proceso de liquidación. Asimismo, se entenderá que el deudor ha incurrido en causal de liquidación, lo que conllevará a la ejecución de los bienes del deudor para satisfacer a los acreedores. En este sentido, el incumplimiento de las obligaciones pactadas en el acuerdo de reorganización no solo afecta la viabilidad del mismo, sino que también puede dar lugar a la pérdida de los beneficios otorgados por la ley, así como a la posibilidad de que los acreedores inicien acciones judiciales para el cobro de sus créditos.


In [12]:
# Decomposition
sub_qs = decompose_query(llm, q3)
print(f'Pregunta compleja descompuesta en {len(sub_qs)} sub-preguntas:')
for i, sq in enumerate(sub_qs, 1):
    print(f'  {i}. {sq}')

Pregunta compleja descompuesta en 3 sub-preguntas:
  1. ¿Cuáles son los requisitos para la reorganización empresarial en Colombia?
  2. ¿Qué características definen el proceso de liquidación empresarial en Colombia?
  3. ¿Cuáles son las principales diferencias entre la reorganización y la liquidación en el contexto de la insolvencia empresarial?


## 5. Agente Completo (ReAct + Reflecting)

**Requiere OPENAI_API_KEY en .env**

In [13]:
from src.tracing.langsmith_setup import configure_langsmith
configure_langsmith()

from src.agent.graph import run_query

# Caso de uso 1: Pregunta directa
result = run_query('¿Qué es la cesación de pagos según la Ley 1116 de 2006?')
print(f'Estrategia: {result["route"]}')
print(f'Reintentos: {result["retry_count"]}')
print(f'Web fallback: {result["used_web_fallback"]}')
print(f'\nRespuesta:\n{result["answer"]}')

[LangSmith] Tracing habilitado. Proyecto: insolvencia-kg-rag
Estrategia: DIRECT
Reintentos: 0
Web fallback: False

Respuesta:
La cesación de pagos, según la Ley 1116 de 2006, se refiere a la incapacidad inminente de una empresa para cumplir con sus obligaciones financieras. Esta situación se considera un requisito fundamental para la apertura de un proceso de reorganización o liquidación obligatoria. La ley establece que la cesación de pagos debe ser evaluada conforme a los artículos 10 y 13 de la misma ley, que delinean los criterios y procedimientos a seguir en tales circunstancias [Fuente: GUIA ORIENTACION final.pdf, p.56].

En este contexto, la cesación de pagos implica que la empresa no puede hacer frente a sus deudas, lo que puede llevar a la solicitud de un proceso de insolvencia para buscar una solución que permita la reestructuración de la empresa o, en su defecto, la liquidación de sus activos para satisfacer a los acreedores [Fuente: 90022.pdf, p.49].


In [14]:
# Caso de uso 2: Pregunta compleja
result = run_query('¿Cuáles son los requisitos y plazos para la reorganización empresarial y qué diferencias hay con la liquidación judicial?')
print(f'Estrategia: {result["route"]}')
print(f'Reintentos: {result["retry_count"]}')
print(f'\nRespuesta:\n{result["answer"]}')

Estrategia: DECOMPOSE
Reintentos: 0

Respuesta:
Para abordar la pregunta sobre los requisitos y plazos para la reorganización empresarial en Colombia, así como las diferencias con la liquidación judicial, se presenta la siguiente información estructurada:

### Requisitos para la Reorganización Empresarial

1. **Solicitud de Iniciación**: La solicitud debe ser presentada por el comerciante que se encuentra en estado de insolvencia. Esta solicitud debe incluir:
   - Los cinco estados financieros básicos correspondientes a los tres últimos ejercicios, debidamente dictaminados. Estos son:
     - Balance General
     - Estado de Resultados
     - Estado de Cambios en el Patrimonio
     - Estado de Cambios en la Situación Financiera
     - Estado de Flujo de Efectivo
   - Notas que acompañen a los estados financieros, las cuales son parte indivisible de ellos [90022.pdf, p.35].

2. **Supuestos de Admisión**: Para que la solicitud sea admitida, se deben cumplir ciertos supuestos, como la cesa

In [15]:
# Caso de uso 3: Pregunta sobre entidades del KG
result = run_query('¿Qué rol cumple la Superintendencia de Sociedades en los procesos de insolvencia?')
print(f'Estrategia: {result["route"]}')
print(f'\nRespuesta:\n{result["answer"]}')

Estrategia: DIRECT

Respuesta:
La Superintendencia de Sociedades desempeña un papel crucial en los procesos de insolvencia empresarial en Colombia, con las siguientes funciones y facultades:

1. **Iniciación de Procesos de Reorganización**: La Superintendencia tiene la facultad de iniciar de oficio el proceso de reorganización en tres situaciones específicas:
   - Cuando una sociedad comercial esté bajo su vigilancia o control.
   - A solicitud expresa de otra autoridad que realice funciones de inspección y vigilancia.
   - En caso de que se presente un proceso de insolvencia de una entidad vinculada que afecte la capacidad de pago de la sociedad en cuestión [ley insolvencia 2006.pdf, p.57].

2. **Designación de Auxiliares de Justicia**: En el inicio del proceso de insolvencia, el juez del concurso designará al promotor o liquidador, quien actúa como auxiliar de la justicia. Esta designación se realiza a partir de una lista elaborada por la Superintendencia de Sociedades [2010.02.15-Bo

## 6. Evaluación del Sistema

In [16]:
from src.evaluation.metrics import evaluate_retrieval

# Ejemplo de evaluación de retrieval
relevant = {'ley-1116-del-27-de-diciembre-de-2006.pdf', 'Cartilla_Ley_1116_ 2006.pdf'}
docs = store.max_marginal_relevance_search('requisitos reorganización', k=10, fetch_k=30)
retrieved = [d.metadata.get('source', '') for d in docs]

metrics = evaluate_retrieval(relevant, retrieved, k_values=[5, 10])
print('Métricas de retrieval:')
for k, v in metrics.items():
    print(f'  {k}: {v:.3f}')

Métricas de retrieval:
  recall@5: 0.000
  precision@5: 0.000
  ndcg@5: 0.000
  recall@10: 0.500
  precision@10: 0.100
  ndcg@10: 0.218
  mrr: 0.167


In [17]:
# LLM como juez
from src.evaluation.llm_judge import llm_judge_score

scores = llm_judge_score(
    llm=llm,
    question='¿Qué es la cesación de pagos?',
    answer=result['answer'],
    context=' '.join(result.get('sources', [])[:3]),
)
print('Puntuación LLM-as-Judge:')
for k, v in scores.items():
    print(f'  {k}: {v}')

Puntuación LLM-as-Judge:
  relevancia: 2.0
  fidelidad: 3.0
  precision_legal: 2.0
  justificacion: La respuesta no aborda directamente la definición de "cesación de pagos", sino que se centra en las funciones de la Superintendencia de Sociedades en el contexto de la insolvencia. Además, aunque menciona aspectos relevantes, no cita de manera precisa las normas o artículos específicos relacionados con la cesación de pagos.
  promedio: 2.3333333333333335
